# 3장 1강 A/B 테스트 핵심 메커니즘과 실험 설계 오류 식별

## Intro

**과제 소개**
프로모션 A/B 데이터를 확인하고, 실험 결과를 신뢰하려면 어떤 조건을 점검해야 하는지 설명합니다. 교안의 개념을 이용해 **판단 → 사례 속 근거 → 이유 → 수정 방향**을 연결하는 것이 목표입니다.

**과제 목표**
- 대조군·실험군과 무작위 배정의 역할을 설명한다.
- 분석 단위·기간·핵심 지표·표본 크기를 구분한다.
- 신규성 효과·표본 오염·조기 종료·SRM을 사례의 근거로 진단한다.
- 핵심 지표와 중요한 보조 지표, 실험 조건을 함께 검토해 결론을 작성한다.

**구성**: 필수 1(중급 초입) → 필수 2(중급) → 과제 1(중급). **필수 1·2는 제출·채점 대상이 아니며, 과제 1만 제출·채점 대상**입니다.

**사용 툴/라이브러리**: Python, Jupyter Notebook, pandas, NumPy, matplotlib.

**학습 범위**: 판단에 필요한 개념은 교안의 1~4절을 사용합니다. 기본 pandas 집계·필터링은 기존 학습 내용을 활용하며, 새로운 통계 검정·표본 크기 계산·세그먼트 분석·수익 지표 설계는 요구하지 않습니다. 데이터 로딩과 그래프 코드는 제공합니다.

### 사용 데이터와 주요 컬럼

| 컬럼 | 의미 | 이번 실습에서의 사용 |
|---|---|---|
| LocationID | 매장 식별자 | 매장 수와 매장별 평균 계산 |
| Promotion | 프로모션 구분 | 1=A, 2=B로 가정; 3은 비교에서 제외 |
| week | 관측 주차 | 주차별 비교 |
| SalesInThousands | 천 단위로 기록된 매출 | 핵심 지표 계산 |
| MarketID / MarketSize / AgeOfStore | 시장·매장 특성 | 이번 실습에서는 분석하지 않음 |

### 분석에 필요한 설명 — 새로운 개념을 추론할 필요가 없도록 제공
- 실제 데이터는 `WA_Marketing-Campaign.csv`입니다. 같은 매장이 4주 동안 반복 기록되어 있습니다.
- 이 실습에서는 **매장을 배정·분석 단위**로 정합니다. 한 매장의 주차별 행이 여러 개여도 서로 다른 매장 여러 곳으로 세지 않습니다.
- 핵심 지표는 **매장별 4주 평균 주간 매출을 계산한 뒤, 그 값을 집단별로 평균한 값**입니다. 4주 누적 매출이 아닙니다.
- A는 기존 프로모션, B는 변경 프로모션이라는 **수업용 역할 가정**입니다. CSV가 실제 기존안·변경안을 알려주는 것은 아닙니다.
- 실제 배정 절차·예정 기간·목표 표본 수·예정 배정 비율은 CSV에 없습니다. 따라서 실제 데이터 문제와 별도로 **교육용 사례**의 조건을 명시합니다. 교육용 수치를 실제 WA 관측 결과와 섞지 않습니다.
- 무작위 배정은 각 대상이 정해진 확률로 배정되게 하는 절차입니다. 결과적인 집단 수가 반드시 같아야 한다는 뜻은 아닙니다.

### 파일 및 제출 안내
- 함께 제공된 실습문제·정답 및 해설 노트북과 기존 `WA_Marketing-Campaign.csv`를 사용합니다.
- 제출 방식: 개인 GitHub 저장소 링크. 예: `product-analysis-assignment-이름/ch03/lesson01.ipynb`, `data/WA_Marketing-Campaign.csv`.
- 제출 노트북에 준비 코드, 과제 1의 코드·실행 결과·요구사항별 답안·Q&A를 포함합니다. 필수 실습 결과를 재사용한다면 필요한 코드도 포함합니다.
- 두 파일 모두 **같은 요구사항 번호**로 구분합니다. 정답 파일의 각 번호에서 코드와 해설을 함께 확인할 수 있습니다.

**파일 구분: 실습문제**

## 실습 준비 — 제공 코드
`DATA_PATH`에 직접 CSV 경로를 입력하거나 `None`을 유지하고 노트북 폴더·`data`·`upload`·`/content`에 CSV를 둡니다. 여러 후보가 발견되면 사용할 파일의 경로를 직접 지정합니다.
아래 코드는 A/B 데이터를 준비하고, 과제에서 사용할 매장별 평균 테이블도 만들어 줍니다. 원본의 Promotion을 새 라벨로 바꾸지 않습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
# 직접 지정하는 예: DATA_PATH = r"C:\Users\사용자명\Downloads\WA_Marketing-Campaign.csv"
DATA_PATH = None
if DATA_PATH is not None:
    csv_path = Path(DATA_PATH).expanduser()
else:
    folders = [Path("."), Path("data"), Path("upload"), Path("/content"), Path("../data")]
    found = list(dict.fromkeys((p / "WA_Marketing-Campaign.csv").resolve()
                              for p in folders if (p / "WA_Marketing-Campaign.csv").is_file()))
    if len(found) != 1:
        raise FileNotFoundError(f"CSV 후보 {len(found)}개. DATA_PATH를 지정하세요. 후보: {found}")
    csv_path = found[0]
campaign = pd.read_csv(csv_path)
campaign_ab = campaign[campaign["Promotion"].isin([1, 2])].copy()
# 같은 매장의 4주 매출을 평균하여 한 매장을 한 행으로 정리합니다.
location_sales = (
    campaign_ab.groupby(["LocationID", "Promotion"], as_index=False)["SalesInThousands"]
    .mean().rename(columns={"SalesInThousands": "four_week_mean_sales"})
)
print("전체 데이터 크기:", campaign.shape)
print("A/B 데이터 크기:", campaign_ab.shape)
display(campaign_ab.head())
display(location_sales.head())

---
## 필수 1. 데이터에서 설계 4대 요소를 구분하기

<aside>
담당자가 “데이터 행 수가 많으니 표본이 충분하고, 두 집단의 수가 비슷하니 무작위 배정도 확인됐다”고 말했습니다. 데이터로 확인되는 사실과 계획서를 봐야 하는 사항을 구분하세요.
이 실습은 제출·채점 대상이 아닙니다.
</aside>

### 문제 1-1. 분석 단위와 무작위 배정 이해

**요구사항**
1. A/B 각각의 행 수와 고유 매장 수, 전체 관측 주차를 출력하세요. 대조군·실험군을 적고, 행 수와 매장 수 중 어느 것을 이번 실습의 표본 수로 사용할지 설명하세요.
2. 설계 4대 요소를 표로 정리하세요. 각 요소에 대해 **이번 분석의 정의 또는 관측값 / CSV만으로 확인하지 못한 사전 계획**을 적으세요. 관측된 4주를 사전에 계획한 4주로 단정하지 않습니다.
3. **다음 실험을 위한 배정 연습**으로 가상의 매장 100개를 A/B에 각각 0.5 확률로 배정하세요. 교안의 `np.random.seed(42)`, `np.random.choice()`를 사용하고 집단별 매장 수를 출력하세요. 정확히 50:50이어야 하는지 설명하세요. 실제 WA 매출에는 이 모의 라벨을 붙이지 않습니다.
4. 아래 Q1·Q2에 답하세요.

**확인 포인트 — 요구사항과 1:1 대응**
1. A/B 역할·행 수·매장 수·관측 주차가 정확하며 매장을 표본 단위로 설명했는가?
2. 네 요소 모두를 작성하고 관측 결과와 사전 계획을 구분했는가?
3. 정해진 확률로 한 번씩 배정하고 결과적인 인원수와 무작위 배정을 구분했는가?
4. Q1·Q2에서 교안의 무작위 배정과 사전 표본 크기 결정 원리를 적용했는가?

**Q&A**
- Q1. 집단별 매장 수가 비슷하면 실제 무작위 배정이 이루어졌다고 말할 수 있나요?
- Q2. 현재 관측된 매장 수만 보고 실험에 필요한 표본 수를 충분히 확보했다고 말할 수 있나요?

### 필수 1 · 요구사항 1 — 역할·표본 수·기간 확인

In [ ]:
role_table = campaign_ab.groupby("Promotion").agg(
    row_count=("SalesInThousands", "size"),
    store_count=("LocationID", "nunique")
)
display(role_table)
print("관측 주차:", sorted(campaign_ab["week"].unique()))


           row_count  store_count
Promotion                        
1                172           43
2                188           47
관측 주차: [1, 2, 3, 4]


**답안 작성**

- 계산·출력 결과: Promotion=1(A) → 행 수 172, 고유 매장 수 43 / Promotion=2(B) → 행 수 188, 고유 매장 수 47. 관측 주차는 1~4주(총 4주).
- 결과 해석 또는 판단 근거: 이번 실습에서는 A(Promotion=1)를 대조군, B(Promotion=2)를 실험군으로 가정합니다. 같은 매장이 4주 동안 반복 기록되어 있으므로 행 수(172, 188)는 실제 표본 수가 아니라 "매장 수 × 4주"가 겹쳐 있는 값입니다. 이 실습의 분석 단위는 매장이므로, 표본 수는 행 수가 아니라 고유 매장 수(A=43개, B=47개)를 사용해야 합니다.


### 필수 1 · 요구사항 2 — 설계 4대 요소 표

**답안 작성**

- 판단 및 근거:

| 설계 요소 | 이번 분석의 정의/관측값 | CSV만으로 확인하지 못한 사전 계획 |
|---|---|---|
| 대조군·실험군 | A=Promotion 1(매장 43개), B=Promotion 2(매장 47개)로 역할을 가정하여 사용 | 실제로 어느 쪽이 기존안(대조군)이고 어느 쪽이 변경안(실험군)인지는 CSV에 기록되어 있지 않음. 역할은 수업용 가정임 |
| 무작위 배정 | CSV에는 배정 결과(어떤 매장이 몇 번 Promotion을 받았는지)만 있음 | 실제로 난수 기반 무작위 배정이 이루어졌는지, 배정 확률이 무엇이었는지, 배정 로그가 있는지는 계획서로 확인해야 함 |
| 분석 단위·기간 | 분석 단위는 매장, 관측된 기간은 1~4주(4주간) | 이 4주가 실험 전에 사전에 계획된 관측 기간인지, 아니면 우연히 4주만 기록된 것인지는 CSV만으로 알 수 없음(관측된 4주 ≠ 계획된 4주) |
| 핵심 지표 | 매장별 4주 평균 주간 매출(SalesInThousands의 매장별 평균) | 이 지표가 실험 전에 사전 지정된 핵심 지표였는지, 목표 수치나 최소 검출 효과크기가 얼마였는지는 계획서가 있어야 확인 가능 |
| 표본 크기 | 현재 확보된 매장 수: A=43개, B=47개 | 실험 전에 필요한 표본 수를 어떤 기준(검정력, 효과크기)으로 산정했는지, 목표 표본 수를 충족했는지는 CSV로 알 수 없음 |

- 수정 방향 또는 추가 확인 사항: "행 수가 많다=표본이 충분하다", "집단 수가 비슷하다=무작위 배정이 확인됐다"는 담당자의 주장은 모두 CSV의 관측 결과와 사전 계획을 혼동한 것입니다. 실제 배정 절차서·사전 표본크기 산정 문서를 확보해 관측값과 대조해야 합니다.


### 필수 1 · 요구사항 3 — 교안 방식으로 무작위 배정 연습

In [ ]:
np.random.seed(42)
practice_stores = np.arange(1, 101)
assignment = np.random.choice(["A", "B"], size=100, p=[0.5, 0.5])
unique, counts = np.unique(assignment, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))


{'A': 53, 'B': 47}


**답안 작성**

- 계산·출력 결과: 100개 매장을 0.5 확률로 A/B에 배정한 결과 A=53개, B=47개.
- 결과 해석 또는 판단 근거: 정확히 50:50이 나올 필요는 없습니다. 무작위 배정은 "각 대상이 정해진 확률(여기서는 0.5)로 독립적으로 배정되는 절차"를 의미하는 것이지, 결과적인 집단 크기가 항상 동일해야 한다는 뜻이 아닙니다. 동전을 100번 던져도 정확히 50번씩 나오지 않는 것처럼, 53:47은 이항분포상 충분히 나타날 수 있는 자연스러운 변동입니다.


### 필수 1 · 요구사항 4 — Q&A

**답안 작성**

- 판단 및 근거:
  - Q1. 아니요. 집단별 매장 수가 비슷한 것은 배정의 "결과"일 뿐이며, 실제로 무작위 배정이 이루어졌는지는 배정 "절차"(난수 사용 여부, 사전에 확정된 배정 확률, 배정 로그 존재 여부)를 확인해야 알 수 있습니다. 담당자가 임의로(수동으로) 매장을 나누었더라도 우연히 개수가 비슷하게 나올 수 있으므로, 개수의 유사성만으로 무작위성을 증명할 수는 없습니다.
  - Q2. 아니요. 현재 관측된 매장 수(A=43, B=47)는 실험이 끝난 뒤 "결과적으로 확보된" 표본 수일 뿐입니다. 실험에 필요한 표본 수는 실험을 시작하기 전에 검출하고자 하는 효과 크기, 유의수준, 검정력 등을 근거로 별도로 산정되어야 하는 값입니다. 사전 계획서 없이 현재 표본 수만 보고 "충분하다/충족했다"고 판단할 수 없습니다.
- 수정 방향 또는 추가 확인 사항: 배정 절차서(난수 시드, 배정 스크립트, 배정 로그)와 사전 표본크기 산정 문서를 확보하여 관측된 숫자와 대조해야 합니다.


---
## 필수 2. 교안의 네 오류를 근거와 수정 방향으로 설명하기

<aside>
오류 이름만 맞히는 데서 끝내지 않습니다. 실제 주차별 매출을 읽고, 조건이 명시된 교육용 사례에서 판단 근거와 수정 방향을 제시하세요.
이 실습은 제출·채점 대상이 아닙니다.
</aside>

### 문제 2-1. 오류 식별과 결과 해석

**요구사항**
1. 제공 코드를 실행해 실제 WA 데이터의 주차별 A/B 평균 매출을 확인하고 `B−A`를 계산하세요. 1주차·4주차 차이를 제시하고, 이 추이만으로 신규성 효과를 확정할 수 있는지 설명하세요. **읽는 방법**: `B−A`가 음수면 그 주의 B 평균이 A보다 낮습니다. 신규성 효과는 새로운 경험에 대한 초기 반응이 시간이 지나며 약해지는 현상이며, 단순한 A/B 우열과 다릅니다.
2. 아래 교육용 사례 N·C·E·S 각각에 대해 **오류 또는 의심 오류 / 사례 속 근거 / 결과 해석에 생기는 문제 / 수정·확인 방향**을 표로 작성하세요. 사례 N·S는 가능성 또는 의심 신호로 표현해도 됩니다. SRM의 통계 계산은 요구하지 않습니다.
3. 실제 WA 데이터에 관해 네 오류 각각을 **확인한 사실 / 추가로 필요한 정보** 표로 정리하세요. 제공된 교차 배정 점검 코드를 실행합니다. 코드가 확인하는 것은 CSV의 Promotion 기록뿐입니다. 교육용 사례를 실제 발생 사실로 쓰지 않습니다.
4. 아래 Q1·Q2에 답하세요.

### 교육용 사례 — 실제 WA 기록이 아닌 별도 상황

| 사례 | 판단을 위해 제공하는 조건 |
|---|---|
| N | 새 프로모션에 대한 반응을 관찰했다. A의 주차별 평균은 50, 50, 50, 50이고 B는 60, 55, 51, 50이었다. 담당자는 첫 주만 근거로 장기 개선 효과가 확정되었다고 말했다. |
| C | 한 매장을 고정 배정하지 않고 매주 다시 배정했다. 같은 매장이 1주차에는 A, 2주차에는 B를 실제 적용받았다. 담당자는 두 집단이 서로 다른 프로모션만 경험했다고 보고했다. |
| E | 시작 전에 4주간 관측하기로 했다. 담당자는 1주차에 B의 평균이 높게 나오자 그 결과를 이유로 종료하고 출시하려 한다. |
| S | 별도 온라인 실험에서 사용자별 A/B 배정 확률을 0.5씩 정했다. 실제 수집 데이터에는 서로 다른 사용자 A 6,500명, B 3,500명이 있었다. 담당자는 배정·수집 상태를 확인하지 않고 매출만 비교하려 한다. |

**확인 포인트 — 요구사항과 1:1 대응**
1. 주차별 차이 계산과 부호 해석이 맞으며 신규성 효과를 확정하지 않았는가?
2. 네 사례 모두에 오류·근거·문제·수정 방향을 연결했는가?
3. 실제 관측과 교육용 조건을 분리하고, 오류별로 필요한 추가 정보를 적었는가?
4. Q1·Q2에서 사전 종료 기준과 표본 오염의 의미를 적용했는가?

**Q&A**
- Q1. 실제 데이터에서 1주와 4주 모두 A의 평균이 높다면, 1주차에 종료했어도 설계상 문제가 없다고 말할 수 있나요?
- Q2. CSV에서 매장별 Promotion 종류 수가 하나이면 모든 표본 오염이 없었다고 말할 수 있나요?

### 필수 2 준비 — 집계·그래프·기록 점검 제공 코드
집계와 그래프 작성 방법 자체는 평가하지 않습니다. 표의 `1`은 A, `2`는 B입니다. 아래 코드로 각 주차의 평균과 기록상 교차 배정 매장 수를 확인하세요.

In [ ]:
weekly_sales = (campaign_ab.groupby(["week", "Promotion"])["SalesInThousands"]
                .mean().unstack("Promotion").sort_index())
# unstack은 Promotion 1·2를 각각 열로 옮겨 두 집단을 나란히 읽게 합니다.
display(weekly_sales.round(3))
weekly_sales.rename(columns={1: "A", 2: "B"}).plot(marker="o", figsize=(8, 4))
plt.title("Weekly mean sales: A and B")
plt.xlabel("Week")
plt.ylabel("SalesInThousands")
plt.xticks([1, 2, 3, 4])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
# 원본 전체를 점검하여 Promotion 3으로 바뀐 기록도 놓치지 않습니다.
promotion_kinds = campaign.groupby("LocationID")["Promotion"].nunique()
print("기록상 여러 Promotion이 있는 매장 수:", int((promotion_kinds > 1).sum()))

### 필수 2 · 요구사항 1 — 실제 주차별 차이 해석

In [ ]:
weekly_sales_diff = weekly_sales.copy()
weekly_sales_diff["B_minus_A"] = weekly_sales_diff[2] - weekly_sales_diff[1]
display(weekly_sales_diff.round(3))
print("1주차 B-A:", round(weekly_sales_diff.loc[1, "B_minus_A"], 3))
print("4주차 B-A:", round(weekly_sales_diff.loc[4, "B_minus_A"], 3))


Promotion       1       2  B_minus_A
week                                
1          58.244  47.730    -10.514
2          56.930  47.583     -9.347
3          58.775  47.722    -11.053
4          58.447  46.283    -12.164
1주차 B-A: -10.514
4주차 B-A: -12.164


**답안 작성**

- 계산·출력 결과: 1주차 B−A = -10.514, 2주차 -9.347, 3주차 -11.053, 4주차 B−A = -12.164. 4주 내내 B의 평균이 A보다 낮게 나타났습니다(부호가 계속 음수).
- 결과 해석 또는 판단 근거: 이 추이만으로 신규성 효과를 확정할 수 없습니다. 신규성 효과는 "새로운 경험에 대한 초기 반응이 시간이 지나며 약해지는 현상"인데, 이 데이터는 B가 애초에 1주차부터 A보다 낮았고, 그 격차도 시간이 갈수록 줄어들지 않고 오히려 -10.514 → -12.164로 더 벌어졌습니다. 즉 "초기에는 컸다가 점점 작아지는" 신규성 효과의 전형적 패턴과 반대이며, 이는 단순한 A/B 우열(B가 계속 A보다 낮음)을 보여줄 뿐 신규성 효과의 근거로 쓸 수 없습니다.


### 필수 2 · 요구사항 2 — 교육용 사례 진단표

**답안 작성**

- 판단 및 근거:

| 사례 | 오류/의심 오류 | 사례 속 근거 | 결과 해석에 생기는 문제 | 수정·확인 방향 |
|---|---|---|---|---|
| N | 신규성 효과 가능성 | B가 60→55→51→50으로 시간이 지나며 A(50)에 점점 가까워짐 | 담당자가 1주차 차이(60 vs 50)만 보고 장기 개선 효과가 "확정"되었다고 말함 — 초기의 큰 차이는 새로움에 대한 일시적 반응일 수 있는데 이를 영구적 개선으로 오인 | 4주보다 더 긴 기간을 추가 관측하여 차이가 완전히 사라지는지 확인하고, 초기 1~2주를 제외한 뒤에도 차이가 유지되는지 재평가 |
| C | 표본 오염(교차 배정) | 같은 매장이 1주차엔 A, 2주차엔 B를 실제로 적용받음(고정 배정이 아님) | 두 집단이 서로 다른 매장 모집단이 아니라 사실상 동일한 매장들이 섞여 있어, 그룹 간 차이가 프로모션 효과인지 순서·이월 효과인지 구분 불가 | 매장을 실험 시작 시 한 번만 배정하고, 같은 매장은 실험이 끝날 때까지 같은 프로모션만 적용받도록 배정 방식을 고정 |
| E | 조기 종료 | 사전에 4주 관측을 계획했으나, 1주차에 B 평균이 높게 나오자 그 결과만으로 종료·출시 결정 | 초기 1주간의 우연한 변동을 실제 효과로 오인할 위험이 크고, 여러 시점을 반복 확인하며 원하는 결과가 나올 때 멈추면 유의성이 부풀려짐 | 사전에 계획한 기간(4주)과 표본 수를 모두 채운 뒤 판단하며, 중간에 확인이 필요하면 사전에 정한 중간분석 규칙(alpha spending 등)을 따름 |
| S | 표본비율불일치(SRM) 의심 | 배정 확률은 A/B 각 0.5였는데, 실제 수집 데이터는 A 6,500명 vs B 3,500명으로 크게 치우침 | 배정·수집·로깅 과정에서 문제가 있었다면 두 집단이 더 이상 무작위로 비교 가능한 대표 표본이 아니게 되어, 매출 차이가 프로모션 효과인지 표본 왜곡 효과인지 알 수 없음 | 배정 로그와 실제 수집 로그를 대조해 유실·오류 구간을 점검하고, 원인이 규명되어 배정 상태가 정상임이 확인된 뒤에 지표 비교를 재개 |

- 수정 방향 또는 추가 확인 사항: N·S는 "가능성/의심 신호"로만 제시했으며 확정적 결론은 아닙니다. C·E는 사례 조건 자체에 설계 위반(교차 배정, 조기 종료)이 명시되어 있어 오류로 판단했습니다.


### 필수 2 · 요구사항 3 — 실제 데이터의 확인 범위

**답안 작성**

- 판단 및 근거: 아래는 실제 `WA_Marketing-Campaign.csv`에서 "확인한 사실"과 앞서 살펴본 교육용 사례(N·C·E·S)를 실제 데이터에 적용하려면 "추가로 필요한 정보"를 구분한 표입니다. 교육용 사례의 조건(수치)은 실제 WA 결과가 아니므로 섞지 않았습니다.

| 오류 | 실제 WA 데이터에서 확인한 사실 | 추가로 필요한 정보 |
|---|---|---|
| 신규성 효과 | 주차별 B−A는 -10.514(1주) → -9.347(2주) → -11.053(3주) → -12.164(4주)로, 격차가 시간이 갈수록 줄어드는 패턴이 아니라 오히려 커짐. 즉 신규성 효과의 전형적 패턴(초기 큰 차이가 점점 작아짐)은 관측되지 않음 | 4주 이후 더 긴 기간의 매출 추이, 프로모션이 실제로 시작된 정확한 시점 기록 |
| 표본 오염(교차 배정) | `campaign.groupby("LocationID")["Promotion"].nunique()` 결과 여러 Promotion이 기록된 매장 수 = 0건. 즉 CSV 기록상으로는 매장마다 Promotion이 하나로 고정되어 있음 | 이는 "기록"만 확인한 것으로, 실제 매장 운영 중 프로모션이 도중에 바뀌었는데 기록에는 반영되지 않은 경우까지는 배제하지 못함 — 현장 운영 로그나 담당자 확인 필요 |
| 조기 종료 | CSV에는 이미 1~4주가 모두 관측된 결과만 있으며, 실험이 실제로 4주 계획대로 끝까지 진행됐는지 도중에 종료 판단이 있었는지는 CSV 자체로는 알 수 없음 | 실험이 사전에 몇 주로 계획되었는지, 중간에 종료를 검토한 시점·의사결정 기록이 있었는지에 대한 계획서·회의록 |
| SRM(표본비율불일치) | A 43개 매장, B 47개 매장으로 100개 중 43:47이며, 사전 목표 배정 비율이 무엇이었는지 모르는 상태에서는 이 정도 차이가 문제인지 판단하기 어려움(S 사례처럼 극단적 치우침은 아님) | 매장 배정 시 사전에 정한 목표 배정 확률(예: 0.5:0.5였는지), 실제 배정 로그와 비교한 SRM 검정 결과 |

- 수정 방향 또는 추가 확인 사항: 표의 "확인한 사실" 열은 CSV의 `Promotion` 기록만으로 확인 가능한 범위이며, 실제로 오류가 있었는지 없었는지를 최종 판단하려면 오른쪽 열의 계획서·로그 자료가 반드시 추가로 필요합니다.


### 필수 2 · 요구사항 4 — Q&A

**답안 작성**

- 판단 및 근거:
  - Q1. 아니요. 1주와 4주 모두 A의 평균이 높다는 것은 "그 두 시점에서의 결과"일 뿐이며, 조기 종료의 문제는 결과의 방향이 맞았는지가 아니라 "사전에 계획한 관측 기간(4주)을 채우지 않고 1주차 결과만으로 종료를 결정했다"는 절차 자체에 있습니다. 우연히 방향이 같게 나왔다고 해서, 초기 변동만 보고 멈추는 의사결정 방식 자체의 위험(우연한 변동을 실제 효과로 오인, 반복 확인에 따른 유의성 부풀림)이 사라지는 것은 아니므로 설계상 문제가 없었다고 말할 수 없습니다.
  - Q2. 아니요. 매장별 Promotion 종류 수가 하나라는 것은 "CSV 기록상" 같은 매장이 여러 프로모션으로 재배정된 적이 없다는 사실만 확인해줄 뿐입니다. 표본 오염에는 이 외에도 매장 직원이 두 프로모션 요소를 섞어서 운영하거나, 고객이 여러 매장을 오가며 서로 다른 프로모션을 모두 접하는 등 CSV에는 기록되지 않는 다른 형태의 오염 가능성이 있습니다. 따라서 이 점검 결과는 "기록상 교차 배정은 없었다"는 것이지 "모든 형태의 표본 오염이 없었다"는 것을 증명하지는 않습니다.


---
## 과제 1. 교안 개념으로 보고서를 고치고 적용 여부 판단하기

<aside>
A를 기존안, B를 변경안으로 가정합니다. 실제 매출을 비교하고 잘못된 보고서를 고친 뒤, 교안의 실험 설계 요소와 결과 해석 원칙으로 다음 행동을 제시하세요.
과제 1(문제 3-1)만 제출·채점 대상입니다.
</aside>

### 문제 3-1. 프로모션 분석 검토 보고서

**잘못된 보고서 초안 — 수정 대상**
1. “A 172개·B 188개 매장을 확보했으므로 필요한 표본 수를 충족했다.”
2. “두 집단의 매장 수가 비슷하므로 무작위 배정이 정상적으로 이루어졌음이 증명되었다.”
3. “1주차에 A가 높았고 4주에도 같으니 1주차 결과만 보고 종료해도 문제가 없다.”
4. “B가 모든 주에서 A보다 낮으므로 신규성 효과는 없다고 확인했다.”
5. “현재 평균에서 B가 낮으므로 다른 조건을 확인할 필요 없이 B는 앞으로도 영구 폐기한다.”

**요구사항**
1. 제공된 `location_sales`로 집단별 평균을 계산하고 `B−A`를 출력하세요. A/B 값·차이·단위를 제시하여 관측 성과를 해석하세요. 이번 강의에서는 유의성 검정이나 인과 효과 확정을 요구하지 않습니다.
2. 초안 1~5번을 각각 **수정 문장 / 실제 수치 또는 교안 근거**로 고치세요. 오류 지적만 하지 말고 보고서에 사용할 수 있는 문장으로 작성합니다.
3. 아래 **교육용 후속 실험 결과**를 보고 B를 바로 전면 적용할지 판단하세요. 핵심 지표 변화·보조 지표 변화·추가 확인을 포함합니다. 주어진 값을 읽어 판단하면 되며 지표 계산식 설계는 요구하지 않습니다.
4. 다음에 실시할 실험의 설계 4대 요소를 표로 작성하세요. 아래 **제공된 계획 조건**을 사용하고 각 요소의 이유를 한 줄씩 설명합니다. 무작위 배정 고정 방식과 중간 결과가 좋을 때의 처리도 덧붙입니다.
5. **실제 WA 결과**에 대한 최종 권고를 4~6문장으로 작성하세요. B 전환 여부·수치 근거·확인하지 못한 실험 조건·다음 확인 행동을 포함하고 Q1·Q2에 답하세요. 요구사항 3의 교육용 수치를 실제 WA 결과에 섞지 않습니다.

### 요구사항 3: 교육용 후속 실험 — WA 실제 결과 아님

| 항목 | A | B |
|---|---:|---:|
| 핵심 지표: 매장별 평균 주간 매출(천 단위) | 50 | 55 |
| 중요한 보조 지표: 환불률 | 2% | 5% |

이 후속 실험은 계획된 기간과 표본 수를 충족했고, 매출 차이는 통계적으로 유의하다는 결과가 제공되었다고 가정합니다. 직접 검정하지 않습니다. 환불률 악화를 어느 정도까지 허용할지는 아직 검토되지 않았고, 배정·수집·실제 적용 점검 결과는 제공되지 않았습니다.

### 요구사항 4: 다음 실험의 제공된 계획 조건 — 새 계획용 가정
- 기존 A와 변경 B의 매출을 비교한다.
- 매장을 기준으로 각각 A/B에 0.5 확률로 한 번 배정하고, 같은 매장은 실험 내내 같은 프로모션을 사용한다.
- 모든 참여 매장이 같은 시점부터 4주간 관측되도록 시작 전에 일정을 확정한다.
- 핵심 지표는 매장별 4주 평균 주간 매출의 집단 평균이다.
- **목표 표본 크기는 실험 전에 별도 담당자가 산정하여 확정한 전체 100개 매장으로 주어진다.** 산정 방법은 이번 강의 범위가 아니며, 100개를 학생이 계산하거나 충분성을 증명할 필요는 없다.
- 계획한 100개 매장을 확보하고 각 매장의 4주 관측을 완료한다. 중간 매출이 좋아 보인다는 이유로 임의 종료하지 않는다.

**확인 포인트 — 요구사항과 1:1 대응**
1. 제공된 매장 단위 평균을 집단별로 비교하고 단위·관측 차이를 정확히 설명했는가?
2. 초안 다섯 문장을 모두 실제 근거와 교안 원리에 맞게 수정했는가?
3. 교육용 핵심·보조 지표를 함께 읽고 실험 조건까지 확인하도록 제안했는가?
4. 제공 조건으로 설계 4대 요소와 이유를 쓰고 배정·종료 원칙을 유지했는가?
5. 실제 WA 근거만으로 권고를 작성하고 Q1·Q2에 답했는가?

**Q&A**
- Q1. 교육용 후속 실험에서 매출 차이가 통계적으로 유의하다고 주어졌는데도 다른 조건을 확인해야 하는 이유는 무엇인가요?
- Q2. 다음 실험에서 전체 100개 매장을 0.5 확률로 배정했는데 53:47로 나뉘었다면, 정확히 50:50으로 맞추기 위해 배정된 매장을 옮겨야 하나요?

### 과제 1 · 요구사항 1 — 실제 매장 단위 평균 비교

In [ ]:
group_means = location_sales.groupby("Promotion")["four_week_mean_sales"].mean()
print(group_means)
b_minus_a = group_means[2] - group_means[1]
print(f"B - A = {b_minus_a:.3f} (단위: 천 달러, 매장별 4주 평균 주간 매출 기준)")


Promotion
1    58.099012
2    47.329415
Name: four_week_mean_sales, dtype: float64
B - A = -10.770 (단위: 천 달러, 매장별 4주 평균 주간 매출 기준)


**답안 작성**

- 계산·출력 결과: A(Promotion=1) 매장별 4주 평균 주간 매출 = 58.099(천 달러), B(Promotion=2) = 47.329(천 달러). B − A = -10.770(천 달러).
- 결과 해석 또는 판단 근거: 매장을 분석 단위로 하여 각 매장의 4주 평균 주간 매출을 구하고, 이를 다시 집단별로 평균한 값입니다(4주 누적 매출이 아님). 관측 결과 실험군(B)이 대조군(A)보다 매장당 주간 매출이 약 10,770달러 낮게 나타났습니다. 이는 현재 데이터에서 관측된 집단 간 차이일 뿐이며, 통계적 유의성 검정이나 인과 효과(프로모션 B가 원인이 되어 매출이 낮아졌다는 것)를 확정하는 결과는 아닙니다.


### 과제 1 · 요구사항 2 — 보고서 다섯 문장 수정

**답안 작성**

보고서 초안 다섯 문장을 아래와 같이 수정합니다.

**1. 원문: "A 172개·B 188개 매장을 확보했으므로 필요한 표본 수를 충족했다."**
- 수정 문장: "A는 43개 매장, B는 47개 매장이 분석 단위(매장) 기준의 표본이다(행 수 172·188은 같은 매장이 4주 동안 반복 기록된 것으로, 표본 수가 아니다). 실험 전에 필요한 표본 수를 어떤 기준으로 산정했는지는 계획서로 확인되지 않아, 현재로서는 충족 여부를 단정할 수 없다."
- 실제 수치/근거: `location_sales` 기준 고유 매장 수 A=43, B=47 (`campaign_ab`의 행 수 172/188과 구분).

**2. 원문: "두 집단의 매장 수가 비슷하므로 무작위 배정이 정상적으로 이루어졌음이 증명되었다."**
- 수정 문장: "두 집단의 매장 수(43 대 47)가 크게 치우치지 않은 것은 배정의 결과일 뿐이며, 실제 무작위 배정이 이루어졌는지는 배정 절차(난수 기반 배정 여부, 사전 확정된 배정 확률, 배정 로그)를 계획서로 확인해야 한다."
- 실제 수치/근거: 교안의 무작위 배정 원리 — 결과적 집단 크기의 유사성과 배정 절차의 무작위성은 별개 개념.

**3. 원문: "1주차에 A가 높았고 4주에도 같으니 1주차 결과만 보고 종료해도 문제가 없다."**
- 수정 문장: "실제로 1주차(B−A=-10.514)부터 4주차(B−A=-12.164)까지 A가 B보다 계속 높게 나타났지만, 이는 4주 관측을 모두 마친 뒤 확인된 결과이다. 사전에 계획한 관측 기간을 채우지 않고 1주차 결과만으로 조기 종료를 결정하는 것은, 결과 방향이 같았다 하더라도 우연한 초기 변동을 실제 효과로 오인할 위험이 있는 별개의 설계상 문제다."
- 실제 수치/근거: `weekly_sales` 주차별 B−A 값.

**4. 원문: "B가 모든 주에서 A보다 낮으므로 신규성 효과는 없다고 확인했다."**
- 수정 문장: "B는 1주차부터 4주차까지 A보다 계속 낮게 나타났고 그 격차도 줄어들지 않고 오히려 커졌다(-10.514 → -12.164). 이는 단순한 A/B 간 우열 관계를 보여줄 뿐이며, 신규성 효과(새로운 경험에 대한 초기 반응이 시간이 지나며 약해지는 현상) 여부를 판단할 근거 자체가 되지 않는다."
- 실제 수치/근거: 신규성 효과의 정의(교안) 및 주차별 B−A 추이.

**5. 원문: "현재 평균에서 B가 낮으므로 다른 조건을 확인할 필요 없이 B는 앞으로도 영구 폐기한다."**
- 수정 문장: "현재 매장 단위 평균에서 B가 A보다 낮게 관측되었으나(B−A=-10.770천 달러), 이는 유의성 검정이나 인과 효과가 확정된 결과가 아니다. 배정 절차, 표본 오염, 조기 종료 여부 등 실험 조건이 아직 확인되지 않았으므로, 이 수치만으로 B를 영구 폐기하는 것은 성급한 결정이다."
- 실제 수치/근거: 과제1 요구사항1의 B−A 계산 결과 및 교안의 결과 해석 시 주의점.


### 과제 1 · 요구사항 3 — 핵심 지표와 보조 지표 함께 판단

**답안 작성**

- 판단 및 근거: 교육용 후속 실험에서 핵심 지표(매장별 평균 주간 매출)는 A=50, B=55로 B가 5(천 달러) 더 높아, 핵심 지표만 보면 B가 개선된 것으로 보입니다. 그러나 중요한 보조 지표인 환불률은 A=2%에서 B=5%로 오히려 악화되었습니다. 매출 증가가 환불 증가를 동반했다면, 실제로 순매출(환불을 제외한 매출) 관점에서는 개선 효과가 상쇄되거나 오히려 손해일 수도 있습니다. 게다가 환불률 악화를 어느 수준까지 허용할지에 대한 사전 기준이 아직 정해지지 않았고, 배정·수집·실제 적용 점검 결과(무작위 배정이 제대로 됐는지, 표본 오염·SRM 문제는 없었는지)도 제공되지 않았습니다.
- 수정 방향 또는 추가 확인 사항: 따라서 매출 차이가 통계적으로 유의하다는 결과만으로 B를 바로 전면 적용하는 것은 시기상조이며, ① 환불률 악화의 허용 기준을 먼저 정하고, ② 배정·수집·적용 상태를 점검하며(SRM 등), ③ 환불을 반영한 순매출(net revenue) 관점에서 다시 판단하고, ④ 장기적으로 재구매율·고객 만족도에 미치는 영향까지 확인한 뒤 전면 적용 여부를 결정해야 합니다.


### 과제 1 · 요구사항 4 — 제공 조건으로 설계 4대 요소 작성

**답안 작성**

- 판단 및 근거: 제공된 계획 조건을 설계 4대 요소로 정리하면 다음과 같습니다.

| 설계 요소 | 다음 실험의 계획 내용 | 이유 |
|---|---|---|
| 대조군·실험군 | 기존안 A와 변경안 B의 매출을 비교 | 기존안 대비 변경안의 효과를 판단하려면 비교 기준이 되는 대조군이 반드시 필요하기 때문 |
| 무작위 배정 | 매장을 기준으로 각각 A/B에 0.5 확률로 한 번 배정하고, 같은 매장은 실험 내내 같은 프로모션을 유지(고정 배정) | 매장별 특성(입지, 규모 등 교란 요인)을 두 집단에 고르게 분산시키고, 필수2에서 확인한 교차 배정(표본 오염) 문제를 원천적으로 방지하기 위함 |
| 분석 단위·기간 | 매장을 분석 단위로 하며, 모든 참여 매장이 같은 시점부터 4주간 관측되도록 시작 전에 일정을 확정 | 매장마다 관측 시작·종료 시점이 다르면 계절성 등 시점 차이가 결과에 섞여 순수한 프로모션 효과 비교가 어려워지기 때문 |
| 핵심 지표 | 매장별 4주 평균 주간 매출의 집단 평균 | 실험 전에 단일 핵심 지표를 미리 정해 두어야, 실험이 끝난 뒤 유리한 지표를 골라 해석하는 오류(체리피킹)를 막을 수 있기 때문 |
| 표본 크기 | 사전에 별도 담당자가 산정하여 확정한 전체 100개 매장을 목표로 함 | 사전에 확정된 표본 수를 채워야 검정력을 확보하고, 결과를 본 뒤 표본 수를 임의로 늘리거나 줄이는 데서 오는 편향을 막을 수 있기 때문 |

- 수정 방향 또는 추가 확인 사항: 무작위 배정은 한 번 정해지면 실험이 끝날 때까지 그대로 고정하며 중간에 재배정하지 않습니다. 또한 계획한 100개 매장을 모두 확보하고 각 매장의 4주 관측을 완료할 때까지는, 중간 매출이 좋아 보이더라도(필수2의 '조기 종료' 오류를 피하기 위해) 임의로 실험을 종료하지 않습니다.


### 과제 1 · 요구사항 5 — 실제 WA 최종 권고와 Q&A

**답안 작성**

- 판단 및 근거(최종 권고, 4~6문장): 실제 WA 매장 단위 데이터에서 대조군 A의 매장별 4주 평균 주간 매출(58.099천 달러)이 실험군 B(47.329천 달러)보다 약 10.770천 달러 높게 관측되어, 현재까지 수집된 결과만으로는 B로 전환할 근거가 없다. 다만 이 수치는 배정 절차(난수 기반 무작위 배정 여부), 표본 오염(교차 배정) 여부, 조기 종료 여부, 목표 표본 수 산정 및 충족 여부 등 실험 조건을 계획서로 확인하지 못한 상태에서 나온 값이므로, 수치가 뚜렷하다고 해서 곧바로 "B가 열등하다"는 인과적 결론을 내리는 것도 성급하다. 다음 확인 행동으로는 ① 실제 배정 절차서·배정 로그를 확보해 무작위성을 점검하고, ② 사전 목표 표본 수 산정 근거를 확인하며, ③ 관측 기간이 사전 계획대로 4주였는지(조기 종료가 없었는지)를 확인해야 한다. 이러한 실험 조건들이 확인되어 현재 데이터의 신뢰성이 뒷받침된다면, 현재로서는 A(기존안)를 유지하는 것이 더 합리적인 판단이며, B는 필요한 조건 확인 후 재평가하는 것이 타당하다.
- 수정 방향 또는 추가 확인 사항:
  - Q1. 교육용 후속 실험에서 매출 차이가 통계적으로 유의하다고 주어졌더라도 다른 조건을 확인해야 하는 이유는, 통계적 유의성은 "이 차이가 우연이 아닐 가능성이 높다"는 것만 말해줄 뿐이기 때문이다. 그 차이가 비즈니스적으로 바람직한지(환불률 악화 등 보조 지표의 부작용을 감수할 가치가 있는지), 그리고 실험이 배정·수집·적용 측면에서 설계대로 정상 수행되었는지는 유의성과는 별개의 문제다. 핵심 지표 하나의 유의성만으로 전면 적용을 결정하면, 부작용이나 실험 절차상의 오류를 놓칠 수 있다.
  - Q2. 아니다. 무작위 배정은 "결과가 정확히 50:50이 되어야 한다"는 뜻이 아니라 "각 매장이 정해진 확률(0.5)로 독립적으로 배정되는 절차" 자체를 의미한다. 100개 매장을 0.5 확률로 배정했을 때 53:47처럼 다소 치우친 결과는 이항분포상 자연스럽게 나타날 수 있는 정상적인 변동이다. 오히려 이미 무작위로 배정이 끝난 매장을 사후에 인위적으로 그룹 간에 옮겨 정확히 50:50을 맞추면, 그 자체로 무작위성이 깨지고 새로운 편향이 발생하므로 그렇게 해서는 안 된다.


---
## 제출 안내 및 마무리
- **과제 1의 요구사항 1~5와 Q&A만 제출·채점 대상**입니다. 필수 1·2는 연습입니다.
- 과제 실행에 필요한 준비 코드와 재사용한 코드는 제출 노트북에 함께 포함합니다.
- 요구사항 번호별로 답안을 작성하고 코드 실행 결과를 남깁니다.
- 실제 WA 결과, 교육용 사례, 다음 실험 계획을 구분합니다.
- 검토 순서: **무엇을 비교했는가 → 설계 조건이 무엇인가 → 어떤 오류를 의심하는가 → 숫자가 무엇을 보여주는가 → 추가 확인 후 무엇을 결정할 것인가**.

### 교안과 문제의 연결

| 교안 | 적용 위치 |
|---|---|
| 1. A/B 핵심 메커니즘 | 필수 1 요구사항 1·3·4, 과제 1 요구사항 2·4·5 |
| 2. 설계 4대 요소 | 필수 1 요구사항 2·4, 과제 1 요구사항 4 |
| 3. 설계 오류 네 가지 | 필수 2 전체, 과제 1 요구사항 2·5 |
| 4. 결과 해석 시 주의점 | 과제 1 요구사항 1·3·5 |